<a href="https://colab.research.google.com/github/Toan-ten/Arxiv-Project-Group10/blob/feat%2Fm1-t02-colab-resume/notebooks/m1_pipeline_thu_thap_va_kiem_toan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Mở trong Colab"/></a>

# Thu thập và kiểm toán dữ liệu arXiv (M1)

Sổ tay chạy cùng một quy trình M1 cho mọi thành viên: lập chỉ mục ứng viên Kaggle, xác minh bài báo và lịch sử phiên bản bằng arXiv, tải mã nguồn, lấy siêu dữ liệu và tài liệu tham khảo từ Semantic Scholar, lưu checkpoint rồi kiểm toán. arXiv xác nhận bài báo và phiên bản.

Trước khi chạy, chọn CPU, kết nối Drive, nhập MSSV theo cấu hình và thêm `KAGGLE_API_TOKEN` cùng `SEMANTIC_SCHOLAR_API_KEY` vào Colab Secrets. Mặc định, quy trình xử lý đủ 5.000 vị trí ID được giao. `COHORT_LIMIT` chỉ giới hạn số lô trong cùng quy trình.

Nếu phiên dừng do hết ngân sách hoặc giới hạn dịch vụ, chờ dịch vụ ổn định rồi chạy lại với cùng MSSV, phạm vi và Drive để tiếp tục từ checkpoint. Không xóa dữ liệu đã lưu hoặc chạy nhiều tiến trình ghi trên cùng corpus.

## 1. Kết nối Drive và kiểm tra commit


In [ ]:
# Ghim mã nguồn vào phiên bản đã kiểm thử.
M1_RUNTIME_COMMIT = "ea0d23a1799953686c64dad7ea884e1cd8db0082"

import os
from pathlib import Path
import re
import subprocess
import sys

loaded_src = sorted(name for name in sys.modules if name == "src" or name.startswith("src."))
if loaded_src:
    raise RuntimeError("Kernel đã nạp mã src trước khi checkout. Chọn Runtime > Restart session rồi chạy lại ô này.")
if os.environ.get("COLAB_GPU") == "1" or os.environ.get("COLAB_TPU_ADDR"):
    raise RuntimeError("Chọn chế độ chỉ dùng CPU trong Runtime trước khi chạy.")

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = Path("/content/Arxiv-Project-Group10")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/Toan-ten/Arxiv-Project-Group10.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

if not isinstance(M1_RUNTIME_COMMIT, str) or not re.fullmatch(r"[0-9a-f]{40}", M1_RUNTIME_COMMIT):
    raise RuntimeError("M1_RUNTIME_COMMIT phải có đủ 40 ký tự thập lục phân.")
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "--detach", M1_RUNTIME_COMMIT], check=True)
current_head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if current_head != M1_RUNTIME_COMMIT:
    raise RuntimeError("Commit đang chạy không khớp M1_RUNTIME_COMMIT.")
if not (REPO_DIR / "src" / "m1_workflow.py").is_file():
    raise RuntimeError("Commit pin chưa có bộ điều phối M1 thống nhất.")
print(f"Commit thực thi: {current_head}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Đã thiết lập môi trường và thư viện phụ thuộc.")

## 2. Chọn thành viên và phạm vi ID

In [ ]:
MSSV = ""  # Nhập MSSV theo config/group10_id_ranges.json.
COHORT_LIMIT = None  # None: toàn bộ 50 lô; 1..50: giới hạn lô để kiểm tra cùng quy trình.


## 3. Chạy quy trình M1

In [ ]:
if not MSSV.strip():
    raise ValueError("Nhập MSSV theo config/group10_id_ranges.json trước khi chạy.")

from google.colab import userdata
from src.m1_workflow import run_m1_member

def _secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

KAGGLE_TOKEN = _secret("KAGGLE_API_TOKEN")
S2_API_KEY = _secret("SEMANTIC_SCHOLAR_API_KEY")
M1_RESULT = run_m1_member(
    MSSV,
    drive_root=Path("/content/drive/MyDrive/ArxivDataGroup10") / MSSV,
    local_root=Path("/content") / f"m1_corpus_{MSSV}",
    kaggle_token=KAGGLE_TOKEN,
    semantic_scholar_api_key=S2_API_KEY,
    cohort_limit=COHORT_LIMIT,
)
KAGGLE_TOKEN = None
S2_API_KEY = None

print("Trạng thái bàn giao:", M1_RESULT["handoff_status"])
print("Kiểm toán:", M1_RESULT["audit"])
print("Lý do dừng:", M1_RESULT["stop_reason"])
print("Báo cáo:", M1_RESULT["report_path"])
print("Bằng chứng đo lường:", M1_RESULT["benchmark_path"])
print("Trạng thái lô:", {
    cohort: {
        "source_complete": details.get("source_complete_papers"),
        "revision_history": details.get("revision_history"),
        "s2_status": details.get("s2_status"),
        "m1_state": details.get("m1_state"),
    }
    for cohort, details in M1_RESULT["cohort_results"].items()
})
